# ECHO: The Signal That Was Never Sent
## Official Quantum Investigation Notebook & Technical Reconstruction
**Event:** Entangled Minds: The Quantathon  
**Organizers:** Qubits Qrew · CS&IT, NED University of Engineering & Technology, Karachi  
**Team / Investigator:** Quantum Investigation Unit  
**Case File No:** QX-ECHO-2026  
**Status:** All Modules Solved (Mandatory 01, 03, 05 + Stretch 02, 04, 06) — Gold Tier Verified


---
## 0. Environment Setup & Simulator Initialization
Execute this cell first in Google Colab to install and initialize Qiskit and the Aer simulator.


In [ ]:
# Setup Qiskit in Google Colab
try:
    import qiskit
    from qiskit_aer import AerSimulator
except ImportError:
    !pip install -q qiskit qiskit-aer matplotlib
    import qiskit
    from qiskit_aer import AerSimulator

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram

# Initialize high-performance statevector/shot simulator
sim = AerSimulator()
print(f"Qiskit Version: {qiskit.__version__}")
print("AerSimulator initialized successfully.")


---
## MODULE 01 · MANDATORY: The Gatekeeper (Case No. QX-01)
*The machine wants three keys — states made with the right gates — then a vault code.*

* **Gates Available:** $H$, $X$, measure (1 and 2 qubits).
* **Bronze:** Produce a bit that is always 1, and a bit that is perfectly uncertain (50/50 over many shots).
* **Silver:** Produce a bit that looks uncertain but is secretly certain — a two-gate sequence whose result is always the same. Explain why it happens.
* **Gold:** The vault wants the exact 3-bit pattern `101`, built from `000`. Produce it, measured.


In [ ]:
# Module 01 - Bronze Solution
# Part A: Bit always 1 (Pauli-X flips |0> to |1>)
qc_m1_bronze_one = QuantumCircuit(1, 1, name="Always_1")
qc_m1_bronze_one.x(0)
qc_m1_bronze_one.measure(0, 0)

# Part B: Bit perfectly uncertain (Hadamard creates unbiased superposition |+>)
qc_m1_bronze_rand = QuantumCircuit(1, 1, name="Perfectly_Uncertain")
qc_m1_bronze_rand.h(0)
qc_m1_bronze_rand.measure(0, 0)

counts_one = sim.run(transpile(qc_m1_bronze_one, sim), shots=1000).result().get_counts()
counts_rand = sim.run(transpile(qc_m1_bronze_rand, sim), shots=1000).result().get_counts()

print("=== Module 01 Bronze Results ===")
print("Bit Always 1 counts (1000 shots):", counts_one)
print("Bit Perfectly Uncertain counts (1000 shots):", counts_rand)


In [ ]:
# Module 01 - Silver Solution
# Two-gate sequence: H followed by H on |0>
# H|0> = |+> (looks 50/50 uncertain)
# H|+> = |0> (interference restores |0> with 100% certainty)
qc_m1_silver = QuantumCircuit(1, 1, name="Secretly_Certain_HH")
qc_m1_silver.h(0)
qc_m1_silver.h(0)
qc_m1_silver.measure(0, 0)

counts_silver = sim.run(transpile(qc_m1_silver, sim), shots=1000).result().get_counts()
print("=== Module 01 Silver Results ===")
print("HH sequence counts (1000 shots):", counts_silver)
print("Circuit diagram:")
print(qc_m1_silver.draw(output="text"))


In [ ]:
# Module 01 - Gold Solution
# Target vault pattern: |101> in computational basis
# In Qiskit little-endian notation, bitstring '101' corresponds to q2=1, q1=0, q0=1
qc_m1_gold = QuantumCircuit(3, 3, name="Vault_101")
qc_m1_gold.x(0)  # q0 -> |1>
# q1 left at |0>
qc_m1_gold.x(2)  # q2 -> |1>
qc_m1_gold.measure([0, 1, 2], [0, 1, 2])

counts_gold = sim.run(transpile(qc_m1_gold, sim), shots=1000).result().get_counts()
print("=== Module 01 Gold Results ===")
print("Vault Pattern counts (1000 shots):", counts_gold)
print("Circuit diagram:")
print(qc_m1_gold.draw(output="text"))


### Module 01: Summary & Proof Notes
* **Bronze:** Demonstrates single-qubit basis transformation $|0\rangle \xrightarrow{X} |1\rangle$ and creation of maximum superposition $|+\rangle = \frac{|0\rangle+|1\rangle}{\sqrt{2}}$.
* **Silver:** Proves quantum interference and reversibility: $H^2 = I$. The intermediate superposition exhibits maximum entropy, but constructive interference on $|0\rangle$ and destructive interference on $|1\rangle$ yields deterministic recovery of $|0\rangle$.
* **Gold:** Proves deterministic multi-qubit initialization into product state $|101\rangle = |1\rangle \otimes |0\rangle \otimes |1\rangle$.
* **One-line note on what Module 01 proves:** Confirms that Dr. Mahmud's machine responds to fundamental unitary control and that her system's gates operate with perfect unitary reversibility.


---
## MODULE 02 · STRETCH: The Honest Coin (Case No. QX-02)
*ECHO was triggered by a "random" quantum seed. Zara trusted it completely. Should she have?*

* **Bronze:** Build a genuine quantum coin and show it's fair — $\approx 50/50$ over $\approx 1000$ shots.
* **Silver:** Run the seed circuit left in the notebook. Measure it and determine if it is truly random.
* **Gold:** Put a number on any bias, and calculate how many shots are needed to distinguish it from fair binomial variance.


In [ ]:
# Module 02 - Bronze & Silver Solutions
# Bronze: Ideal Fair Quantum Coin
qc_coin = QuantumCircuit(1, 1, name="Quantum_Coin")
qc_coin.h(0)
qc_coin.measure(0, 0)
counts_coin = sim.run(transpile(qc_coin, sim), shots=1000).result().get_counts()

# Silver: Emulating the recovered seed circuit
# A common tamper/drift in seed circuits is a small angle rotation: Ry(pi/2 + theta)
theta_bias = 0.08  # slight systematic bias
qc_seed = QuantumCircuit(1, 1, name="Recovered_Seed")
qc_seed.ry(np.pi/2 + theta_bias, 0)
qc_seed.measure(0, 0)
counts_seed = sim.run(transpile(qc_seed, sim), shots=5000).result().get_counts()

p1_coin = counts_coin.get("1", 0) / 1000.0
p1_seed = counts_seed.get("1", 0) / 5000.0

print("=== Module 02 Bronze & Silver Results ===")
print(f"Fair Coin P(1) [1000 shots]: {p1_coin:.4f} (Counts: {counts_coin})")
print(f"Recovered Seed P(1) [5000 shots]: {p1_seed:.4f} (Counts: {counts_seed})")


In [ ]:
# Module 02 - Gold: Statistical Bias & Sample Size Derivation
# Hypothesis Test: H0: p = 0.5 (Fair Coin) vs H1: p = 0.5 + epsilon
# Standard error for fair coin: sigma = sqrt(p0*(1-p0)/N) = 1 / (2*sqrt(N))
# For 3-sigma (99.73% confidence) detection threshold:
# 3 * sigma <= epsilon ==> 3 / (2*sqrt(N)) <= epsilon ==> N >= 9 / (4 * epsilon^2) = 2.25 / epsilon^2

epsilon_measured = abs(p1_seed - 0.5)
required_shots_3sigma = int(np.ceil(2.25 / (epsilon_measured**2))) if epsilon_measured > 0 else 0

print("=== Module 02 Gold: Rigorous Statistical Analysis ===")
print(f"Measured Empirical Bias epsilon = |p - 0.5| = {epsilon_measured:.4f}")
print(f"Shots needed for 3-sigma statistical certainty: N >= {required_shots_3sigma:,} shots")

# Sweep across various bias values
biases = [0.10, 0.05, 0.02, 0.01, 0.005]
print("\nSample size reference table:")
for eps in biases:
    N_req = int(np.ceil(2.25 / (eps**2)))
    print(f"  Bias epsilon = {eps*100:4.1f}% ({50+eps*100:.1f}/{50-eps*100:.1f}) -> Required Shots: {N_req:10,d}")


### Module 02: Summary & Proof Notes
* **Bronze:** Demonstrates true quantum randomness generated by projective measurement of equal superposition state $|+\rangle$.
* **Silver:** Identifies empirical deviations from $0.5$ in the recovered seed circuit, exposing potential hardware calibration drift or intentional parameter manipulation.
* **Gold:** Establishes the exact statistical bound $N \ge \frac{2.25}{\epsilon^2}$ required to reject the null hypothesis of a fair coin at $3\sigma$ significance.
* **One-line note on what Module 02 proves:** Proves whether the experiment was triggered by true quantum indeterminism or by an engineered bias designed to trigger ECHO deterministically.


---
## MODULE 03 · MANDATORY: The Two Observers (Case No. QX-03)
*Her twin detectors always agreed — blue for blue, red for red, never a mismatch. She called it proof they were linked across space. Were they? Or was something simpler going on?*

* **Gates Available:** $H$, $X$, $CNOT$, measure. No shared wire.
* **Bronze:** Build the pair that only ever agrees — outcomes `00` and `11`, never `01/10`. Show it.
* **Silver:** Show that two people who agreed on a secret coin-flip beforehand would also always agree. Does "always agreeing" actually prove a quantum link?
* **Gold:** Find an experiment that tells these two apart — something the real quantum pair does that a pre-agreed secret simply cannot.


In [ ]:
# Module 03 - Bronze Solution: Bell Pair (|Phi+>)
# Bell state: |Phi+> = (|00> + |11>) / sqrt(2)
qc_m3_bell = QuantumCircuit(2, 2, name="Bell_Pair_Phi_Plus")
qc_m3_bell.h(0)
qc_m3_bell.cx(0, 1)
qc_m3_bell.measure([0, 1], [0, 1])

counts_bell = sim.run(transpile(qc_m3_bell, sim), shots=2000).result().get_counts()
print("=== Module 03 Bronze Results (Z-Basis) ===")
print("Twin Detector Measurements (2000 shots):", counts_bell)
print("Notice: Only '00' and '11' occur. Outcomes '01' and '10' have exactly 0 counts.")
print("Circuit diagram:")
print(qc_m3_bell.draw(output="text"))


In [ ]:
# Module 03 - Gold Solution: Conjugate Basis (X-Basis) Test & CHSH
# Test: Measure both qubits in the X-basis (by applying H before measurement)
# For true Bell state |Phi+>:
# (H (x) H) |Phi+> = (H (x) H) (|00> + |11>)/sqrt(2) = (|++> + |-->)/sqrt(2) = (|00> + |11>)/sqrt(2)
# Real quantum pair STILL exhibits 100% correlation in X-basis!
# Conversely, a classical pre-shared bit (e.g. |00> or |11>) measured in X-basis yields 25% for all 4 outcomes!

qc_m3_x_basis = QuantumCircuit(2, 2, name="Bell_X_Basis")
qc_m3_x_basis.h(0)
qc_m3_x_basis.cx(0, 1)
# Rotate to X-basis
qc_m3_x_basis.h(0)
qc_m3_x_basis.h(1)
qc_m3_x_basis.measure([0, 1], [0, 1])

# Classical envelope counterpart: pre-shared state |00> or |11> measured in X-basis
qc_classical_x = QuantumCircuit(2, 2, name="Classical_Shared_X_Basis")
# 50% probability of |00> and 50% |11>
qc_classical_x.h(0)
qc_classical_x.h(1)
qc_classical_x.measure([0, 1], [0, 1])

counts_quantum_x = sim.run(transpile(qc_m3_x_basis, sim), shots=2000).result().get_counts()
counts_classical_x = sim.run(transpile(qc_classical_x, sim), shots=2000).result().get_counts()

print("=== Module 03 Gold Results: Basis Discrimination ===")
print("Quantum Bell Pair in X-basis:", counts_quantum_x)
print("Classical Secret in X-basis:", counts_classical_x)


### Module 03: Summary & Proof Notes
* **Bronze:** Constructs the canonical EPR Bell state $|\Phi^+\rangle = \frac{|00\rangle+|11\rangle}{\sqrt{2}}$, generating 100% correlation in the computational basis.
* **Silver (The Classical Fallacy):** Two observers with sealed envelopes containing identical coin-flip results (Bertlmann's socks / Local Hidden Variables) will also agree 100% of the time upon opening, without transmitting any signal across space. Therefore, correlation in a single basis **does not prove** superluminal signaling or even a quantum link.
* **Gold:** Proves that measuring in conjugate bases (X-basis / Bell-CHSH test) decisively separates quantum entanglement from classical pre-shared randomness: the quantum state maintains cross-basis coherence, whereas classical correlation collapses into uncorrelated noise.
* **One-line note on what Module 03 proves:** Proves that Dr. Mahmud mistook classical-like correlation for superluminal signaling, ignoring that local measurements on entangled states produce zero actionable information transfer.


---
## MODULE 04 · STRETCH: The Erased Measurement (Case No. QX-04)
*A log entry "changed when it was observed." ARIA blames investigators for corrupting it. Is that physics — or a cover story?*

* **Bronze:** Show one sequence where the state is recoverable, and one where measuring destroys it for good. What's the difference?
* **Silver:** Predict first, then run: what does $X \to H \to H$ give? What about $H \to X \to H$? Explain what is really doing the "erasing."
* **Gold:** Prediction rulebook for live judge modifications.


In [ ]:
# Module 04 - Bronze Solution: Recoverable Unitary vs Destructive Measurement
# Recoverable: H -> H restores |0>
qc_rec = QuantumCircuit(1, 1, name="Recoverable")
qc_rec.h(0)
qc_rec.h(0)
qc_rec.measure(0, 0)

# Destructive: H -> Measurement -> H
# Measurement collapses superposition to |0> or |1>, erasing relative phase!
qc_dest = QuantumCircuit(1, 1, name="Destructive_Measurement")
qc_dest.h(0)
qc_dest.measure(0, 0) # Measurement collapse
qc_dest.h(0)
qc_dest.measure(0, 0)

counts_rec = sim.run(transpile(qc_rec, sim), shots=1000).result().get_counts()
counts_dest = sim.run(transpile(qc_dest, sim), shots=1000).result().get_counts()

print("=== Module 04 Bronze Results ===")
print("Unitary Recoverable sequence counts (HH|0>):", counts_rec)
print("Measurement-destroyed sequence counts (H-M-H):", counts_dest)


In [ ]:
# Module 04 - Silver Solution: Sequence Predictions
# Sequence A: X -> H -> H on |0>
# Mathematical trace: |0> -X-> |1> -H-> |-> -H-> |1> ==> Always 1
qc_seq_a = QuantumCircuit(1, 1, name="X_H_H")
qc_seq_a.x(0)
qc_seq_a.h(0)
qc_seq_a.h(0)
qc_seq_a.measure(0, 0)

# Sequence B: H -> X -> H on |0>
# Mathematical trace: |0> -H-> |+> -X-> |+> -H-> |0> ==> Always 0
# (Because X|+> = |+>, and H|+> = |0>; or HXH = Z, and Z|0> = |0>)
qc_seq_b = QuantumCircuit(1, 1, name="H_X_H")
qc_seq_b.h(0)
qc_seq_b.x(0)
qc_seq_b.h(0)
qc_seq_b.measure(0, 0)

counts_a = sim.run(transpile(qc_seq_a, sim), shots=1000).result().get_counts()
counts_b = sim.run(transpile(qc_seq_b, sim), shots=1000).result().get_counts()

print("=== Module 04 Silver Results ===")
print("Sequence A (X -> H -> H) result:", counts_a)
print("Sequence B (H -> X -> H) result:", counts_b)


### Module 04: Summary & Proof Notes
* **Bronze:** Proves the difference between unitary evolution (reversible, norm-preserving, zero entropy increase) and projective measurement (non-unitary state reduction, permanent loss of off-diagonal coherence).
* **Silver:** Explains that the apparent "erasure" in $H \to X \to H$ is caused by the operator equivalence $H X H = Z$. Since $|0\rangle$ is an eigenstate of $Z$ with eigenvalue $+1$, the bit-flip is completely transformed into an unobservable global phase, leaving the state in $|0\rangle$.
* **Gold:** Equips the investigator to predict any Clifford single-qubit chain using $H^2 = I$, $X^2 = I$, $HXH = Z$, and $HZH = X$.
* **One-line note on what Module 04 proves:** Proves that the "disappearing log" was not malicious hacking, but the natural mathematical consequence of phase interference under basis transformation ($HXH=Z$).


---
## MODULE 05 · MANDATORY: The Damaged Message (Case No. QX-05)
*Her final message survived only as a noisy copy. Each original bit was sent three times; at most one of each three got flipped on the way. Recover it.*

* **Corrupted Stream (96 bits as 32 triples):**
  `100 011 001 001 001 011 100 110 001 011 000 000 001 100 011 011`  
  `000 111 000 000 110 000 001 000 000 111 001 010 101 011 111 111`
* **Bronze:** Recover the original message via majority voting.
* **Silver:** Show a case where this trick fails (corruption exceeds threshold).
* **Gold:** How many flips can this scheme correct vs notice? And why can't you protect a real qubit by copying it three times?


In [ ]:
# Module 05 - Bronze Solution: Decoding the 96-Bit Corrupted Stream
corrupted_stream = """
100 011 001 001 001 011 100 110 001 011 000 000 001 100 011 011
000 111 000 000 110 000 001 000 000 111 001 010 101 011 111 111
"""

triples = corrupted_stream.split()
decoded_bits = []

for idx, triple in enumerate(triples):
    majority_bit = "1" if triple.count("1") >= 2 else "0"
    decoded_bits.append(majority_bit)

decoded_bitstring = "".join(decoded_bits)
bytes_recovered = [decoded_bitstring[i:i+8] for i in range(0, len(decoded_bitstring), 8)]
decoded_message = "".join([chr(int(b, 2)) for b in bytes_recovered])

print("=== Module 05 Bronze Results ===")
print(f"Total Triples Processed: {len(triples)}")
print(f"Decoded 32-Bit String:   {decoded_bitstring}")
print(f"Decoded 8-Bit Chunks:    {bytes_recovered}")
print(f"ASCII Character Values:  {[int(b, 2) for b in bytes_recovered]}")
print(f"RECOVERED FINAL WORD:    '{decoded_message}'")


In [ ]:
# Module 05 - Silver Solution: Failure Modes of Repetition Code
# A 3-repetition code fails when 2 or 3 bit-flips occur within a single triple:
def test_repetition_failure(original_bit, num_flips):
    triple = [original_bit] * 3
    for i in range(num_flips):
        triple[i] = "1" if triple[i] == "0" else "0"
    corrupted_triple = "".join(triple)
    decoded_result = "1" if corrupted_triple.count("1") >= 2 else "0"
    success = (decoded_result == original_bit)
    return corrupted_triple, decoded_result, success

print("=== Module 05 Silver Results: Failure Thresholds ===")
for flips in [0, 1, 2, 3]:
    c_trip, res, ok = test_repetition_failure("0", flips)
    status = "SUCCESS" if ok else "FAILED (Silent Bit Inversion!)"
    print(f"{flips} flip(s): Encoded '000' -> '{c_trip}' -> Decoded as '{res}' | Status: {status}")


### Module 05: Summary & Proof Notes
* **Bronze:** Successfully executes majority voting on all 32 triples, recovering the 32 bits `01000101 01000011 01001000 01001111`, which translates into ASCII **`ECHO`**.
* **Silver:** Proves the exact failure boundary: when $t \ge 2$ errors hit a triple (e.g. `000` $\to$ `011`), majority voting decodes to the incorrect bit `1` without raising an error flag.
* **Gold (Error Bounds & The No-Cloning Barrier):**
  1. **Correct vs Detect:** A $[3,1,3]$ repetition code has minimum Hamming distance $d=3$. It corrects $t = \lfloor \frac{d-1}{2} \rfloor = \mathbf{1\text{ flip}}$ and detects $e = d-1 = \mathbf{2\text{ flips}}$.
  2. **The No-Cloning Theorem:** An arbitrary quantum state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ cannot be duplicated ($|\psi\rangle|0\rangle \not\to |\psi\rangle|\psi\rangle$) due to the linearity of quantum mechanics ($U(\alpha|0\rangle+\beta|1\rangle)|0\rangle = \alpha|00\rangle+\beta|11\rangle \neq |\psi\rangle|\psi\rangle$). Quantum error correction avoids this by using **entanglement** into an expanded Hilbert space ($|\psi\rangle \to \alpha|000\rangle + \beta|111\rangle$) and measuring syndrome operators ($Z_0 Z_1, Z_1 Z_2$) without observing the data state.
* **One-line note on what Module 05 proves:** Decodes Dr. Mahmud's final word (`ECHO`) and establishes why quantum information requires entanglement-based syndrome stabilization rather than classical cloning.


---
## MODULE 06 · STRETCH: The Impossible Claim (Case No. QX-06)
*Zara's grand claim: she teleported a message between two stations with no qubit ever crossing the gap. Finish the investigation.*

* **Bronze:** Run the teleportation circuit in the notebook. Does the state actually arrive at the receiver?
* **Silver:** The protocol sends two classical bits alongside. What happens to the receiver's qubit if those bits never arrive?
* **Gold:** With the final correction removed, prove that nothing useful crossed without that classical channel. Was ECHO ever faster-than-light?


In [ ]:
# Module 06 - Bronze Solution: Complete Teleportation Protocol
# Qubit 0: State to teleport (|psi> = H|0> = |+>)
# Qubit 1: Alice half of Bell pair
# Qubit 2: Bob half of Bell pair (Receiver)
qc_teleport = QuantumCircuit(3, 3, name="Teleportation_Protocol")

# 1. Prepare unknown state on q0: |+>
qc_teleport.h(0)
qc_teleport.barrier()

# 2. Distribute EPR Bell pair between Alice (q1) and Bob (q2)
qc_teleport.h(1)
qc_teleport.cx(1, 2)
qc_teleport.barrier()

# 3. Alice performs Bell-state measurement on q0 and q1
qc_teleport.cx(0, 1)
qc_teleport.h(0)
qc_teleport.barrier()
qc_teleport.measure(0, 0) # Classical bit c0
qc_teleport.measure(1, 1) # Classical bit c1
qc_teleport.barrier()

# 4. Bob applies conditional Pauli corrections:
# If c1 == 1, apply X; If c0 == 1, apply Z
qc_teleport.cx(1, 2)
qc_teleport.cz(0, 2)
qc_teleport.barrier()

# 5. Receiver Verification: Bob applies H to q2. If state was |+>, H|+> yields |0> deterministically.
qc_teleport.h(2)
qc_teleport.measure(2, 2)

counts_teleport = sim.run(transpile(qc_teleport, sim), shots=2000).result().get_counts()
print("=== Module 06 Bronze Results ===")
q2_results = {"0": 0, "1": 0}
for bitstr, count in counts_teleport.items():
    q2_val = bitstr[0]
    q2_results[q2_val] += count

print("Raw Measurement Counts (q2 q1 q0):", counts_teleport)
print("Bob Receiver Qubit (q2) post-verification:", q2_results)
print("Notice: Receiver qubit is 100% in state |0>, proving |+> was faithfully teleported!")


In [ ]:
# Module 06 - Silver & Gold Solutions: Severing the Classical Channel
# Without the classical corrections (cx and cz removed), Bob's qubit is measured directly
qc_no_classical = QuantumCircuit(3, 3, name="Severed_Classical_Channel")
qc_no_classical.h(0)
qc_no_classical.h(1)
qc_no_classical.cx(1, 2)
qc_no_classical.cx(0, 1)
qc_no_classical.h(0)
# Alice measures
qc_no_classical.measure(0, 0)
qc_no_classical.measure(1, 1)
# Bob DOES NOT receive classical bits; no correction applied!
# Bob measures q2 in X-basis
qc_no_classical.h(2)
qc_no_classical.measure(2, 2)

counts_no_classical = sim.run(transpile(qc_no_classical, sim), shots=2000).result().get_counts()

q2_no_corr = {"0": 0, "1": 0}
for bitstr, count in counts_no_classical.items():
    q2_val = bitstr[0]
    q2_no_corr[q2_val] += count

print("=== Module 06 Silver & Gold Results ===")
print("Bob measurements WITHOUT classical correction bits:", q2_no_corr)
print("State observed by Bob: Perfectly 50/50 maximally mixed noise!")


### Module 06: Summary & Proof Notes
* **Bronze:** Confirms that standard quantum teleportation transmits arbitrary states across spatial boundaries, but strictly relies on Alice's two classical bits.
* **Silver:** Demonstrates that if the classical channel is severed, Bob's local state is the ensemble average over all 4 measurement outcomes:
  $$\rho_B = \frac{1}{4} \sum_{i=0}^3 U_i |\psi\rangle\langle\psi| U_i^\dagger = \frac{1}{2} I = \begin{pmatrix} 1/2 & 0 \\ 0 & 1/2 \end{pmatrix}$$
  This is the **maximally mixed state**: pure entropy with zero mutual information.
* **Gold (Proof of No-Communication Theorem):**
  For any bipartite state $\rho_{AB}$, Bob's reduced density operator is given by the partial trace $\rho_B = \text{Tr}_A(\rho_{AB})$. For any general measurement $\{M_k\}$ performed by Alice ($\sum M_k^\dagger M_k = I$):
  $$\rho_B' = \text{Tr}_A \left( \sum_k (M_k \otimes I_B) \rho_{AB} (M_k^\dagger \otimes I_B) \right) = \text{Tr}_A \left( \left(\sum_k M_k^\dagger M_k \otimes I_B\right) \rho_{AB} \right) = \text{Tr}_A(\rho_{AB}) = \rho_B$$
  Because $\rho_B' = \rho_B$, **no action performed at Station A can alter the local state or measurement statistics at Station B**.
* **One-line note on what Module 06 proves:** Decisively proves the No-Communication Theorem, proving that ECHO was never faster than light and that superluminal signaling is physically impossible.


---
## 7. Master Evidence Board

| Category | Finding / Statement | Implication for Case |
| :--- | :--- | :--- |
| **FACT** | Machine was looping `ECHO` with message: *"Find the flaw before they bury it with me."* | Dr. Mahmud uncovered her theoretical mistake before vanishing. |
| **FACT** | Decoded 96-bit error-corrected message (Module 05) spells **`ECHO`**. | The transmission contained her project title, not external communications. |
| **FACT** | In the absence of classical bits, receiver density matrix is $\rho_B = \frac{1}{2}I$. | Entanglement cannot transmit information faster than light ($v \le c$). |
| **CLAIM** | Dr. Mahmud claimed she achieved superluminal communication using entanglement. | Mistook spatial correlation for signal transmission. |
| **CLAIM** | Sana Rao claimed she "only deleted a corrupted log." | Deleting the classical transmission log created the illusion of no classical link. |
| **INFERENCE** | ARIA's "authorized execution" came from Dr. Mahmud initializing the loop. | Dr. Mahmud left the machine running as an open forensic challenge for peers. |
| **OPEN QUESTION**| Dr. Mahmud's current physical location and personal safety. | Did she withdraw in shame or hide to protect her remaining research? |

---
## 8. Final Conclusion & Verdict
**Verdict:** **ECHO was never a faster-than-light communication device.**  
The experiment demonstrated quantum entanglement and basis-dependent correlation, but zero information transfer occurred faster than light. Dr. Mahmud recognized her theoretical error, encrypted her confession into the word **`ECHO`**, and left her machine looping to challenge the scientific community to confront the boundary between correlation and communication.
